# Eye Disease Classification — Custom CNN recovered experiment

This notebook consolidates the complete set of **recovered** training histories, model metadata, final evaluation, confusion matrix, and TTA comparison. It reads committed artifacts and does not retrain the model or invent missing results. The original end-to-end training source notebook was not present in the download; the two recovered source fragments are preserved beside this notebook.

In [ ]:
from pathlib import Path
import csv, json, sys
import matplotlib.pyplot as plt
import pandas as pd
import torch

cwd = Path.cwd().resolve()
candidates = [cwd / 'Custom_CNN', cwd, cwd.parent]
ROOT = next((p for p in candidates if (p / 'results/experiment_summary.json').exists()), None)
assert ROOT is not None, 'Run from the combined repository root, Custom_CNN, or notebooks directory'
sys.path.insert(0, str(ROOT))
from src.model import CustomCNN, load_checkpoint
print('Repository:', ROOT)

## Dataset manifests

Only the exact recovered split CSVs are included. Images are intentionally excluded.

In [ ]:
splits = {name: pd.read_csv(ROOT / 'splits' / f'{name}.csv') for name in ['train', 'val', 'test']}
split_summary = pd.DataFrame({
    name: {'rows': len(frame), 'classes': frame.unified_label.nunique(), 'sources': frame.dataset.nunique()}
    for name, frame in splits.items()
}).T
assert split_summary['rows'].to_dict() == {'train': 8287, 'val': 1184, 'test': 2368}
display(split_summary)

## Recorded training experiments

The export contains histories for the baseline, two learning-rate fine-tuning stages, focal-loss/strong-augmentation stage, and balanced-sampling/augmentation stage. Epoch numbers restart within each saved stage.

In [ ]:
history_files = {
    'Baseline': 'baseline_history.json',
    'LR 0.0001': 'lr_0001_history.json',
    'LR 0.00005': 'lr_00005_history.json',
    'Focal + strong aug': 'focal_strong_aug_history.json',
    'Balanced sampling + aug': 'balanced_history.json',
}
histories = {}
for label, filename in history_files.items():
    histories[label] = pd.read_json(ROOT / 'results' / 'training' / filename)
    histories[label].insert(0, 'stage', label)
training_table = pd.concat(histories.values(), ignore_index=True)
display(training_table)
plt.figure(figsize=(16, 5))
plt.imshow(plt.imread(ROOT / 'figures' / 'training_history.png'))
plt.axis('off'); plt.show()

## Selected checkpoint

The state dictionary fixes every parameterized layer name and shape. The accompanying model definition uses conventional placements for stateless ReLU/pooling and dropout operations; exact placement cannot be proven without the missing original source. The strict load and forward-pass checks below verify structural compatibility, not historical preprocessing equivalence.

In [ ]:
checkpoint_path = ROOT / 'checkpoints' / 'best_custom_cnn.pth'
model, checkpoint = load_checkpoint(checkpoint_path)
metadata = {key: value for key, value in checkpoint.items()
            if key not in {'model_state_dict', 'optimizer_state_dict'}}
display(metadata)
with torch.inference_mode():
    logits = model(torch.zeros(1, 3, 224, 224))
assert logits.shape == (1, 8)
print(f'Strict load passed; output shape: {tuple(logits.shape)}')

## Final held-out test results (without TTA)

In [ ]:
with (ROOT / 'results' / 'final_test' / 'final_test_results.json').open() as handle:
    final = json.load(handle)
print(f"Accuracy: {final['test_accuracy']:.6f}")
print(f"Macro-F1: {final['test_macro_f1']:.6f}")
print(f"Samples: {final['test_samples']}")
report = pd.read_csv(ROOT / 'results' / 'final_test' / 'classification_report.csv')
display(report)
plt.figure(figsize=(10, 8))
plt.imshow(plt.imread(ROOT / 'results' / 'final_test' / 'final_confusion_matrix.png'))
plt.axis('off'); plt.show()

## Recorded TTA comparison (validation only)

These values come from `experiment_summary.json`. The recovered newer source fragment also preserves a separate 8-view report whose variables depend on missing earlier cells; it is retained for provenance but is not executed here. The final test was evaluated without TTA.

In [ ]:
tta = pd.read_csv(ROOT / 'results' / 'tta' / 'tta_validation_comparison.csv')
display(tta)
plt.figure(figsize=(10, 5))
plt.imshow(plt.imread(ROOT / 'figures' / 'tta_validation_comparison.png'))
plt.axis('off'); plt.show()

## What is and is not reproducible

Recovered and validated: exact split manifests, all recorded stage histories, selected checkpoint tensors/metadata, final test report and confusion matrix, and summarized validation TTA scores. Missing: the original complete training notebook, exact preprocessing/augmentation and sampler code, loss/optimizer construction, and definitions of every TTA view. Consequently this package supports transparent artifact inspection and strict checkpoint loading, but it does not claim bit-for-bit reproduction of training or inference.